### Reading Patient Table

In [0]:
df = spark.read.table('medical_catalog.bronze.patients')
display(df.limit(10))


### Convert Column to Lowercase and SnakeCase

In [0]:
# Convert columns to snakecase
for col in df.columns:
    new_col = col.lower().replace(' ', '_')
    if new_col != col:
        df = df.withColumnRenamed(col, new_col)
display(df.limit(10))


### TypeCasting BithDate

In [0]:
from pyspark.sql.functions import col, try_to_date

df = df.withColumn('birthdate__', try_to_date(col("birthdate__"), "yyyy-MM-dd")) \
       .withColumnRenamed("birthdate__", "birthdate")


display(df.limit(10))


### Drop Columns which are Unnecessary

In [0]:
df = df.drop('maiden','race','birth_place')
display(df.limit(10))

### Remove Numbers Attached with the First and Last Name

In [0]:
from pyspark.sql.functions import regexp_replace

df = df.withColumn("first", regexp_replace(col("first"), r"\d+", "")) \
       .withColumn("last", regexp_replace(col("last"), r"\d+", ""))

display(df.limit(10))

##### Merge Both First and Last Columns as **Fullname**

In [0]:
from pyspark.sql.functions import col, concat_ws, trim

# Combine columns with a space separator
df = df.withColumn("fullname", trim(concat_ws(" ", col("first"), col("last"))))

display(df.limit(10))


### Drop the First and Last Columns

In [0]:
df = df.drop('first','last','suffix')
display(df.limit(10))

#### Fill Null Value 

In [0]:
from pyspark.sql.functions import when, lit

df = df.withColumn("death_date", when(col("death_date").isNull(), lit("9999-12-31")).otherwise(col("death_date")))

display(df.limit(10))

#### Write the Patients df into Tables

In [0]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medical_catalog.silver.patients")
